# Non-local thermistor: PINN vs VPINN — quick run

$$-u''(x) = \frac{\lambda\, e^{-u}}{\left(\int_0^1 e^{-u}\,dx\right)^2},\qquad u(0)=u(1)=0$$

Unique solution for $\lambda < 8$, none for $\lambda \ge 8$ (thermal runaway).
This notebook runs the **short** version of the benchmark (`run_all.py --quick`,
about 5–10 min on a Colab CPU) and shows the figures. The full benchmark is `python run_all.py`.

In [ ]:
# Clone the repository (edit the URL if you use a fork)
!git clone https://github.com/MaximeAuger/nonlocal-thermistor-pinn.git
%cd nonlocal-thermistor-pinn
!pip -q install -r requirements.txt

## 1. Check the exact solution and the quadrature

In [ ]:
!python -m pytest tests -q
!cd src && python exact_solution.py

## 2. Single training runs ($\lambda = 7$)

In [ ]:
import sys; sys.path.insert(0, "src")
from pinn_solver import train_pinn
from vpinn_solver import train_vpinn
for train, extra in [(train_pinn, {}), (train_vpinn, {"n_test": 60})]:
    r = train({"lam": 7.0, "n_adam": 1500, "n_lbfgs": 500, **extra}, verbose=False)
    m = r["metrics"]
    print(f"{r['method']:5s}  L2 = {m['L2']:.2e}   rel. err. I[u] = {m['I_rel']:.2e}   time = {r['time']:.0f}s")

## 3. Quick benchmark (all experiments, reduced settings)

In [ ]:
!cd src && python run_all.py --quick --nproc 2

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results_quick/results_summary.md", encoding="utf-8").read()))
for f in ["fig1_accuracy_vs_lambda", "fig2_nonexistence_training", "fig3_lambda_sweep",
          "fig4_local_vs_nonlocal", "fig5_ablations"]:
    display(Image(f"results_quick/{f}.png"))